# IW39 — Centro 4128 — Datalake

**Tabela:** `dev_procurement.corp_curated.tbl_ds_ind_iw39`
**Domínio:** Ordens de manutencao (PM)
**Filtro do cenário:** `cod_centro_planejamento_manutencao = '4128'`
**Colunas:** 61 · **Clustering declarado:** `cod_ordem`

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente com o filtro do centro embutido. Não há widget, view temporária nem ordem obrigatória.

## Objetivo

Extrair e caracterizar **toda** a base do centro 4128 para comparação com o extrato do SAP.

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados da tabela |
| 2 | Volumetria e representatividade do cenário |
| 3 | Confirmação do filtro |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| 6 | Preenchimento de todas as colunas |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 9 | Perfil numérico |
| **10** | **Totais para conciliação com o SAP** |
| 11 | Datas |
| 12 | Códigos e zeros à esquerda |
| **13** | **Chaves normalizadas para join** |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| 17 | Freshness |
| 18 | Análises específicas |
| **19** | **EXTRAÇÃO COMPLETA** |
| 20 | Resumo do cenário |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 14.


## 1. Metadados da tabela

In [ ]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_ind_iw39;

In [ ]:
-- Formato, tamanho e particoes (falha se nao for Delta)
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_ind_iw39;

In [ ]:
-- Ultimas gravacoes (falha se for view)
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_ind_iw39 LIMIT 20;

## 2. Volumetria e representatividade

Quanto o centro 4128 representa do total da tabela.

In [ ]:
-- 2. VOLUMETRIA DO CENARIO
SELECT
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_ind_iw39)                                   AS linhas_tabela_toda,
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128')                               AS linhas_centro_4128,
  ROUND(100.0 * (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128')
              / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_ind_iw39), 4)                  AS pct_do_total,
  (SELECT COUNT(DISTINCT `cod_centro_planejamento_manutencao`) FROM dev_procurement.corp_curated.tbl_ds_ind_iw39)                     AS centros_na_tabela;

## 3. Confirmação do filtro

Confirma que o valor `4128` existe e que não há variação de formato
(espaços, zeros à esquerda) que faça o filtro perder linhas silenciosamente.

**Se retornar mais de uma linha, o filtro `= '4128'` está incompleto.**

In [ ]:
-- 3. O FILTRO PEGOU TUDO?
SELECT CAST(`cod_centro_planejamento_manutencao` AS STRING)                    AS valor_bruto,
       length(CAST(`cod_centro_planejamento_manutencao` AS STRING))            AS comprimento,
       COUNT(*)                                   AS linhas
FROM dev_procurement.corp_curated.tbl_ds_ind_iw39
WHERE regexp_replace(trim(CAST(`cod_centro_planejamento_manutencao` AS STRING)), '^0+', '') = '4128'
   OR trim(CAST(`cod_centro_planejamento_manutencao` AS STRING)) = '4128'
GROUP BY CAST(`cod_centro_planejamento_manutencao` AS STRING), length(CAST(`cod_centro_planejamento_manutencao` AS STRING))
ORDER BY linhas DESC;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional
multiplicando as linhas.

In [ ]:
-- 4. GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'),
g AS (
  SELECT 'cod_ordem' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_ordem` FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128')
UNION ALL
  SELECT 'cod_ordem + num_nota' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_ordem`, `num_nota` FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128')
UNION ALL
  SELECT 'cod_ordem + cod_equipamento' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_ordem`, `cod_equipamento` FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128')
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `cod_ordem`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [ ]:
-- 5. CHAVES DUPLICADAS
SELECT `cod_ordem`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
GROUP BY `cod_ordem`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [ ]:
-- 5.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH cen AS (
  SELECT * FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
),
dup AS (
  SELECT `cod_ordem` FROM cen GROUP BY `cod_ordem` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM cen c JOIN dup ON c.`cod_ordem` <=> dup.`cod_ordem`
),
agg AS (
  SELECT `cod_ordem`,
         COUNT(DISTINCT `tp_grupo_planejamento_mnt`) AS `tp_grupo_planejamento_mnt`,
         COUNT(DISTINCT `cod_centro_trabalho_principal`) AS `cod_centro_trabalho_principal`,
         COUNT(DISTINCT `tp_ordem`) AS `tp_ordem`,
         COUNT(DISTINCT `cod_prioridade`) AS `cod_prioridade`,
         COUNT(DISTINCT `dt_criacao`) AS `dt_criacao`,
         COUNT(DISTINCT `dt_base_inicio`) AS `dt_base_inicio`,
         COUNT(DISTINCT `dt_base_fim`) AS `dt_base_fim`,
         COUNT(DISTINCT `num_nota`) AS `num_nota`,
         COUNT(DISTINCT `desc_breve`) AS `desc_breve`,
         COUNT(DISTINCT `cod_revisao`) AS `cod_revisao`,
         COUNT(DISTINCT `cod_localizacao`) AS `cod_localizacao`,
         COUNT(DISTINCT `tp_grupo_processamento`) AS `tp_grupo_processamento`,
         COUNT(DISTINCT `cod_usuario_criacao`) AS `cod_usuario_criacao`,
         COUNT(DISTINCT `vl_custo_total_planejado`) AS `vl_custo_total_planejado`,
         COUNT(DISTINCT `cod_usuario_modificacao`) AS `cod_usuario_modificacao`,
         COUNT(DISTINCT `cod_plano_manutencao`) AS `cod_plano_manutencao`,
         COUNT(DISTINCT `cod_equipamento`) AS `cod_equipamento`,
         COUNT(DISTINCT `cod_centro_custo_responsavel`) AS `cod_centro_custo_responsavel`,
         COUNT(DISTINCT `cod_centro_custo`) AS `cod_centro_custo`,
         COUNT(DISTINCT `dt_encerramento_tecnico`) AS `dt_encerramento_tecnico`,
         COUNT(DISTINCT `dt_inicio_programado`) AS `dt_inicio_programado`,
         COUNT(DISTINCT `dt_fim_programado`) AS `dt_fim_programado`,
         COUNT(DISTINCT `dt_referencia`) AS `dt_referencia`,
         COUNT(DISTINCT `dh_referencia`) AS `dh_referencia`,
         COUNT(DISTINCT `dt_ultima_modificacao`) AS `dt_ultima_modificacao`,
         COUNT(DISTINCT `dt_liberacao_real`) AS `dt_liberacao_real`,
         COUNT(DISTINCT `dt_fim_confirmado_ordem`) AS `dt_fim_confirmado_ordem`,
         COUNT(DISTINCT `dh_base_fim`) AS `dh_base_fim`,
         COUNT(DISTINCT `dh_fim_confirmado_ordem`) AS `dh_fim_confirmado_ordem`,
         COUNT(DISTINCT `dt_inicio_real`) AS `dt_inicio_real`,
         COUNT(DISTINCT `dh_inicio_real`) AS `dh_inicio_real`,
         COUNT(DISTINCT `dh_base_inicio`) AS `dh_base_inicio`,
         COUNT(DISTINCT `cod_elemento_pep`) AS `cod_elemento_pep`,
         COUNT(DISTINCT `cod_pep_ordem`) AS `cod_pep_ordem`,
         COUNT(DISTINCT `tp_categoria_ordem`) AS `tp_categoria_ordem`,
         COUNT(DISTINCT `cod_centro_planejamento_manutencao`) AS `cod_centro_planejamento_manutencao`,
         COUNT(DISTINCT `cod_id_objeto_centro_trabalho`) AS `cod_id_objeto_centro_trabalho`,
         COUNT(DISTINCT `tp_grupo_lista_operacoes`) AS `tp_grupo_lista_operacoes`,
         COUNT(DISTINCT `cod_variante_lista_operacoes`) AS `cod_variante_lista_operacoes`,
         COUNT(DISTINCT `num_serie`) AS `num_serie`,
         COUNT(DISTINCT `cod_material`) AS `cod_material`,
         COUNT(DISTINCT `desc_produto_material`) AS `desc_produto_material`,
         COUNT(DISTINCT `cod_conjunto`) AS `cod_conjunto`,
         COUNT(DISTINCT `cod_moeda`) AS `cod_moeda`,
         COUNT(DISTINCT `cod_esquema_calculo_custos`) AS `cod_esquema_calculo_custos`,
         COUNT(DISTINCT `cod_empresa`) AS `cod_empresa`,
         COUNT(DISTINCT `cod_divisao`) AS `cod_divisao`,
         COUNT(DISTINCT `cod_centro_lucro`) AS `cod_centro_lucro`,
         COUNT(DISTINCT `cod_area_contabilidade_custos`) AS `cod_area_contabilidade_custos`,
         COUNT(DISTINCT `cod_ordem_cliente`) AS `cod_ordem_cliente`,
         COUNT(DISTINCT `num_item_pedido_venda`) AS `num_item_pedido_venda`,
         COUNT(DISTINCT `cod_diagrama_rede_rede_superior`) AS `cod_diagrama_rede_rede_superior`,
         COUNT(DISTINCT `cod_ordem_tem_texto_descritivo`) AS `cod_ordem_tem_texto_descritivo`,
         COUNT(DISTINCT `ind_marcacao_eliminacao`) AS `ind_marcacao_eliminacao`,
         COUNT(DISTINCT `cod_rua_endereco`) AS `cod_rua_endereco`,
         COUNT(DISTINCT `cod_regiao_endereco`) AS `cod_regiao_endereco`,
         COUNT(DISTINCT `nm_cidade_endereco`) AS `nm_cidade_endereco`,
         COUNT(DISTINCT `cod_pais_endereco`) AS `cod_pais_endereco`,
         COUNT(DISTINCT `cod_postal_endereco`) AS `cod_postal_endereco`,
         COUNT(DISTINCT `num_telefone_endereco`) AS `num_telefone_endereco`
  FROM d GROUP BY `cod_ordem`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(60,
    'tp_grupo_planejamento_mnt', MAX(`tp_grupo_planejamento_mnt`),
    'cod_centro_trabalho_principal', MAX(`cod_centro_trabalho_principal`),
    'tp_ordem', MAX(`tp_ordem`),
    'cod_prioridade', MAX(`cod_prioridade`),
    'dt_criacao', MAX(`dt_criacao`),
    'dt_base_inicio', MAX(`dt_base_inicio`),
    'dt_base_fim', MAX(`dt_base_fim`),
    'num_nota', MAX(`num_nota`),
    'desc_breve', MAX(`desc_breve`),
    'cod_revisao', MAX(`cod_revisao`),
    'cod_localizacao', MAX(`cod_localizacao`),
    'tp_grupo_processamento', MAX(`tp_grupo_processamento`),
    'cod_usuario_criacao', MAX(`cod_usuario_criacao`),
    'vl_custo_total_planejado', MAX(`vl_custo_total_planejado`),
    'cod_usuario_modificacao', MAX(`cod_usuario_modificacao`),
    'cod_plano_manutencao', MAX(`cod_plano_manutencao`),
    'cod_equipamento', MAX(`cod_equipamento`),
    'cod_centro_custo_responsavel', MAX(`cod_centro_custo_responsavel`),
    'cod_centro_custo', MAX(`cod_centro_custo`),
    'dt_encerramento_tecnico', MAX(`dt_encerramento_tecnico`),
    'dt_inicio_programado', MAX(`dt_inicio_programado`),
    'dt_fim_programado', MAX(`dt_fim_programado`),
    'dt_referencia', MAX(`dt_referencia`),
    'dh_referencia', MAX(`dh_referencia`),
    'dt_ultima_modificacao', MAX(`dt_ultima_modificacao`),
    'dt_liberacao_real', MAX(`dt_liberacao_real`),
    'dt_fim_confirmado_ordem', MAX(`dt_fim_confirmado_ordem`),
    'dh_base_fim', MAX(`dh_base_fim`),
    'dh_fim_confirmado_ordem', MAX(`dh_fim_confirmado_ordem`),
    'dt_inicio_real', MAX(`dt_inicio_real`),
    'dh_inicio_real', MAX(`dh_inicio_real`),
    'dh_base_inicio', MAX(`dh_base_inicio`),
    'cod_elemento_pep', MAX(`cod_elemento_pep`),
    'cod_pep_ordem', MAX(`cod_pep_ordem`),
    'tp_categoria_ordem', MAX(`tp_categoria_ordem`),
    'cod_centro_planejamento_manutencao', MAX(`cod_centro_planejamento_manutencao`),
    'cod_id_objeto_centro_trabalho', MAX(`cod_id_objeto_centro_trabalho`),
    'tp_grupo_lista_operacoes', MAX(`tp_grupo_lista_operacoes`),
    'cod_variante_lista_operacoes', MAX(`cod_variante_lista_operacoes`),
    'num_serie', MAX(`num_serie`),
    'cod_material', MAX(`cod_material`),
    'desc_produto_material', MAX(`desc_produto_material`),
    'cod_conjunto', MAX(`cod_conjunto`),
    'cod_moeda', MAX(`cod_moeda`),
    'cod_esquema_calculo_custos', MAX(`cod_esquema_calculo_custos`),
    'cod_empresa', MAX(`cod_empresa`),
    'cod_divisao', MAX(`cod_divisao`),
    'cod_centro_lucro', MAX(`cod_centro_lucro`),
    'cod_area_contabilidade_custos', MAX(`cod_area_contabilidade_custos`),
    'cod_ordem_cliente', MAX(`cod_ordem_cliente`),
    'num_item_pedido_venda', MAX(`num_item_pedido_venda`),
    'cod_diagrama_rede_rede_superior', MAX(`cod_diagrama_rede_rede_superior`),
    'cod_ordem_tem_texto_descritivo', MAX(`cod_ordem_tem_texto_descritivo`),
    'ind_marcacao_eliminacao', MAX(`ind_marcacao_eliminacao`),
    'cod_rua_endereco', MAX(`cod_rua_endereco`),
    'cod_regiao_endereco', MAX(`cod_regiao_endereco`),
    'nm_cidade_endereco', MAX(`nm_cidade_endereco`),
    'cod_pais_endereco', MAX(`cod_pais_endereco`),
    'cod_postal_endereco', MAX(`cod_postal_endereco`),
    'num_telefone_endereco', MAX(`num_telefone_endereco`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada **neste centro**.

Uma coluna pode ter dado na tabela toda e estar vazia no centro 4128 — ou o contrário.
Por isso a varredura é feita sobre o recorte, não sobre a base completa.

In [ ]:
-- 6. PREENCHIMENTO NO CENTRO 4128
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'),
perf AS (
  SELECT stack(61,
    'cod_ordem', 'string', COUNT_IF(`cod_ordem` IS NULL), COUNT_IF(`cod_ordem` IS NOT NULL AND lower(trim(`cod_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_ordem`) RLIKE '^0+([.,]0+)?$'),
    'tp_grupo_planejamento_mnt', 'string', COUNT_IF(`tp_grupo_planejamento_mnt` IS NULL), COUNT_IF(`tp_grupo_planejamento_mnt` IS NOT NULL AND lower(trim(`tp_grupo_planejamento_mnt`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_grupo_planejamento_mnt`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_trabalho_principal', 'string', COUNT_IF(`cod_centro_trabalho_principal` IS NULL), COUNT_IF(`cod_centro_trabalho_principal` IS NOT NULL AND lower(trim(`cod_centro_trabalho_principal`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_trabalho_principal`) RLIKE '^0+([.,]0+)?$'),
    'tp_ordem', 'string', COUNT_IF(`tp_ordem` IS NULL), COUNT_IF(`tp_ordem` IS NOT NULL AND lower(trim(`tp_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_ordem`) RLIKE '^0+([.,]0+)?$'),
    'cod_prioridade', 'string', COUNT_IF(`cod_prioridade` IS NULL), COUNT_IF(`cod_prioridade` IS NOT NULL AND lower(trim(`cod_prioridade`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_prioridade`) RLIKE '^0+([.,]0+)?$'),
    'dt_criacao', 'string', COUNT_IF(`dt_criacao` IS NULL), COUNT_IF(`dt_criacao` IS NOT NULL AND lower(trim(`dt_criacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_criacao`) RLIKE '^0+([.,]0+)?$'),
    'dt_base_inicio', 'string', COUNT_IF(`dt_base_inicio` IS NULL), COUNT_IF(`dt_base_inicio` IS NOT NULL AND lower(trim(`dt_base_inicio`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_base_inicio`) RLIKE '^0+([.,]0+)?$'),
    'dt_base_fim', 'string', COUNT_IF(`dt_base_fim` IS NULL), COUNT_IF(`dt_base_fim` IS NOT NULL AND lower(trim(`dt_base_fim`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_base_fim`) RLIKE '^0+([.,]0+)?$'),
    'num_nota', 'string', COUNT_IF(`num_nota` IS NULL), COUNT_IF(`num_nota` IS NOT NULL AND lower(trim(`num_nota`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_nota`) RLIKE '^0+([.,]0+)?$'),
    'desc_breve', 'string', COUNT_IF(`desc_breve` IS NULL), COUNT_IF(`desc_breve` IS NOT NULL AND lower(trim(`desc_breve`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_breve`) RLIKE '^0+([.,]0+)?$'),
    'cod_revisao', 'string', COUNT_IF(`cod_revisao` IS NULL), COUNT_IF(`cod_revisao` IS NOT NULL AND lower(trim(`cod_revisao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_revisao`) RLIKE '^0+([.,]0+)?$'),
    'cod_localizacao', 'string', COUNT_IF(`cod_localizacao` IS NULL), COUNT_IF(`cod_localizacao` IS NOT NULL AND lower(trim(`cod_localizacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_localizacao`) RLIKE '^0+([.,]0+)?$'),
    'tp_grupo_processamento', 'string', COUNT_IF(`tp_grupo_processamento` IS NULL), COUNT_IF(`tp_grupo_processamento` IS NOT NULL AND lower(trim(`tp_grupo_processamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_grupo_processamento`) RLIKE '^0+([.,]0+)?$'),
    'cod_usuario_criacao', 'string', COUNT_IF(`cod_usuario_criacao` IS NULL), COUNT_IF(`cod_usuario_criacao` IS NOT NULL AND lower(trim(`cod_usuario_criacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_usuario_criacao`) RLIKE '^0+([.,]0+)?$'),
    'vl_custo_total_planejado', 'double', COUNT_IF(`vl_custo_total_planejado` IS NULL), 0L, COUNT_IF(`vl_custo_total_planejado` = 0),
    'cod_usuario_modificacao', 'string', COUNT_IF(`cod_usuario_modificacao` IS NULL), COUNT_IF(`cod_usuario_modificacao` IS NOT NULL AND lower(trim(`cod_usuario_modificacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_usuario_modificacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_plano_manutencao', 'string', COUNT_IF(`cod_plano_manutencao` IS NULL), COUNT_IF(`cod_plano_manutencao` IS NOT NULL AND lower(trim(`cod_plano_manutencao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_plano_manutencao`) RLIKE '^0+([.,]0+)?$'),
    'cod_equipamento', 'string', COUNT_IF(`cod_equipamento` IS NULL), COUNT_IF(`cod_equipamento` IS NOT NULL AND lower(trim(`cod_equipamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_equipamento`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_custo_responsavel', 'string', COUNT_IF(`cod_centro_custo_responsavel` IS NULL), COUNT_IF(`cod_centro_custo_responsavel` IS NOT NULL AND lower(trim(`cod_centro_custo_responsavel`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_custo_responsavel`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_custo', 'string', COUNT_IF(`cod_centro_custo` IS NULL), COUNT_IF(`cod_centro_custo` IS NOT NULL AND lower(trim(`cod_centro_custo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_custo`) RLIKE '^0+([.,]0+)?$'),
    'dt_encerramento_tecnico', 'string', COUNT_IF(`dt_encerramento_tecnico` IS NULL), COUNT_IF(`dt_encerramento_tecnico` IS NOT NULL AND lower(trim(`dt_encerramento_tecnico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_encerramento_tecnico`) RLIKE '^0+([.,]0+)?$'),
    'dt_inicio_programado', 'string', COUNT_IF(`dt_inicio_programado` IS NULL), COUNT_IF(`dt_inicio_programado` IS NOT NULL AND lower(trim(`dt_inicio_programado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_inicio_programado`) RLIKE '^0+([.,]0+)?$'),
    'dt_fim_programado', 'string', COUNT_IF(`dt_fim_programado` IS NULL), COUNT_IF(`dt_fim_programado` IS NOT NULL AND lower(trim(`dt_fim_programado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_fim_programado`) RLIKE '^0+([.,]0+)?$'),
    'dt_referencia', 'string', COUNT_IF(`dt_referencia` IS NULL), COUNT_IF(`dt_referencia` IS NOT NULL AND lower(trim(`dt_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_referencia`) RLIKE '^0+([.,]0+)?$'),
    'dh_referencia', 'string', COUNT_IF(`dh_referencia` IS NULL), COUNT_IF(`dh_referencia` IS NOT NULL AND lower(trim(`dh_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dh_referencia`) RLIKE '^0+([.,]0+)?$'),
    'dt_ultima_modificacao', 'string', COUNT_IF(`dt_ultima_modificacao` IS NULL), COUNT_IF(`dt_ultima_modificacao` IS NOT NULL AND lower(trim(`dt_ultima_modificacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^0+([.,]0+)?$'),
    'dt_liberacao_real', 'string', COUNT_IF(`dt_liberacao_real` IS NULL), COUNT_IF(`dt_liberacao_real` IS NOT NULL AND lower(trim(`dt_liberacao_real`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_liberacao_real`) RLIKE '^0+([.,]0+)?$'),
    'dt_fim_confirmado_ordem', 'string', COUNT_IF(`dt_fim_confirmado_ordem` IS NULL), COUNT_IF(`dt_fim_confirmado_ordem` IS NOT NULL AND lower(trim(`dt_fim_confirmado_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_fim_confirmado_ordem`) RLIKE '^0+([.,]0+)?$'),
    'dh_base_fim', 'string', COUNT_IF(`dh_base_fim` IS NULL), COUNT_IF(`dh_base_fim` IS NOT NULL AND lower(trim(`dh_base_fim`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dh_base_fim`) RLIKE '^0+([.,]0+)?$'),
    'dh_fim_confirmado_ordem', 'string', COUNT_IF(`dh_fim_confirmado_ordem` IS NULL), COUNT_IF(`dh_fim_confirmado_ordem` IS NOT NULL AND lower(trim(`dh_fim_confirmado_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dh_fim_confirmado_ordem`) RLIKE '^0+([.,]0+)?$'),
    'dt_inicio_real', 'string', COUNT_IF(`dt_inicio_real` IS NULL), COUNT_IF(`dt_inicio_real` IS NOT NULL AND lower(trim(`dt_inicio_real`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_inicio_real`) RLIKE '^0+([.,]0+)?$'),
    'dh_inicio_real', 'string', COUNT_IF(`dh_inicio_real` IS NULL), COUNT_IF(`dh_inicio_real` IS NOT NULL AND lower(trim(`dh_inicio_real`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dh_inicio_real`) RLIKE '^0+([.,]0+)?$'),
    'dh_base_inicio', 'string', COUNT_IF(`dh_base_inicio` IS NULL), COUNT_IF(`dh_base_inicio` IS NOT NULL AND lower(trim(`dh_base_inicio`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dh_base_inicio`) RLIKE '^0+([.,]0+)?$'),
    'cod_elemento_pep', 'string', COUNT_IF(`cod_elemento_pep` IS NULL), COUNT_IF(`cod_elemento_pep` IS NOT NULL AND lower(trim(`cod_elemento_pep`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_elemento_pep`) RLIKE '^0+([.,]0+)?$'),
    'cod_pep_ordem', 'string', COUNT_IF(`cod_pep_ordem` IS NULL), COUNT_IF(`cod_pep_ordem` IS NOT NULL AND lower(trim(`cod_pep_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_pep_ordem`) RLIKE '^0+([.,]0+)?$'),
    'tp_categoria_ordem', 'string', COUNT_IF(`tp_categoria_ordem` IS NULL), COUNT_IF(`tp_categoria_ordem` IS NOT NULL AND lower(trim(`tp_categoria_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_categoria_ordem`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_planejamento_manutencao', 'string', COUNT_IF(`cod_centro_planejamento_manutencao` IS NULL), COUNT_IF(`cod_centro_planejamento_manutencao` IS NOT NULL AND lower(trim(`cod_centro_planejamento_manutencao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_planejamento_manutencao`) RLIKE '^0+([.,]0+)?$'),
    'cod_id_objeto_centro_trabalho', 'string', COUNT_IF(`cod_id_objeto_centro_trabalho` IS NULL), COUNT_IF(`cod_id_objeto_centro_trabalho` IS NOT NULL AND lower(trim(`cod_id_objeto_centro_trabalho`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_id_objeto_centro_trabalho`) RLIKE '^0+([.,]0+)?$'),
    'tp_grupo_lista_operacoes', 'string', COUNT_IF(`tp_grupo_lista_operacoes` IS NULL), COUNT_IF(`tp_grupo_lista_operacoes` IS NOT NULL AND lower(trim(`tp_grupo_lista_operacoes`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_grupo_lista_operacoes`) RLIKE '^0+([.,]0+)?$'),
    'cod_variante_lista_operacoes', 'string', COUNT_IF(`cod_variante_lista_operacoes` IS NULL), COUNT_IF(`cod_variante_lista_operacoes` IS NOT NULL AND lower(trim(`cod_variante_lista_operacoes`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_variante_lista_operacoes`) RLIKE '^0+([.,]0+)?$'),
    'num_serie', 'string', COUNT_IF(`num_serie` IS NULL), COUNT_IF(`num_serie` IS NOT NULL AND lower(trim(`num_serie`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_serie`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'desc_produto_material', 'string', COUNT_IF(`desc_produto_material` IS NULL), COUNT_IF(`desc_produto_material` IS NOT NULL AND lower(trim(`desc_produto_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_produto_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_conjunto', 'string', COUNT_IF(`cod_conjunto` IS NULL), COUNT_IF(`cod_conjunto` IS NOT NULL AND lower(trim(`cod_conjunto`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_conjunto`) RLIKE '^0+([.,]0+)?$'),
    'cod_moeda', 'string', COUNT_IF(`cod_moeda` IS NULL), COUNT_IF(`cod_moeda` IS NOT NULL AND lower(trim(`cod_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda`) RLIKE '^0+([.,]0+)?$'),
    'cod_esquema_calculo_custos', 'string', COUNT_IF(`cod_esquema_calculo_custos` IS NULL), COUNT_IF(`cod_esquema_calculo_custos` IS NOT NULL AND lower(trim(`cod_esquema_calculo_custos`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_esquema_calculo_custos`) RLIKE '^0+([.,]0+)?$'),
    'cod_empresa', 'string', COUNT_IF(`cod_empresa` IS NULL), COUNT_IF(`cod_empresa` IS NOT NULL AND lower(trim(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa`) RLIKE '^0+([.,]0+)?$'),
    'cod_divisao', 'string', COUNT_IF(`cod_divisao` IS NULL), COUNT_IF(`cod_divisao` IS NOT NULL AND lower(trim(`cod_divisao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_divisao`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_lucro', 'string', COUNT_IF(`cod_centro_lucro` IS NULL), COUNT_IF(`cod_centro_lucro` IS NOT NULL AND lower(trim(`cod_centro_lucro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_lucro`) RLIKE '^0+([.,]0+)?$'),
    'cod_area_contabilidade_custos', 'string', COUNT_IF(`cod_area_contabilidade_custos` IS NULL), COUNT_IF(`cod_area_contabilidade_custos` IS NOT NULL AND lower(trim(`cod_area_contabilidade_custos`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_area_contabilidade_custos`) RLIKE '^0+([.,]0+)?$'),
    'cod_ordem_cliente', 'string', COUNT_IF(`cod_ordem_cliente` IS NULL), COUNT_IF(`cod_ordem_cliente` IS NOT NULL AND lower(trim(`cod_ordem_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_ordem_cliente`) RLIKE '^0+([.,]0+)?$'),
    'num_item_pedido_venda', 'string', COUNT_IF(`num_item_pedido_venda` IS NULL), COUNT_IF(`num_item_pedido_venda` IS NOT NULL AND lower(trim(`num_item_pedido_venda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_pedido_venda`) RLIKE '^0+([.,]0+)?$'),
    'cod_diagrama_rede_rede_superior', 'string', COUNT_IF(`cod_diagrama_rede_rede_superior` IS NULL), COUNT_IF(`cod_diagrama_rede_rede_superior` IS NOT NULL AND lower(trim(`cod_diagrama_rede_rede_superior`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_diagrama_rede_rede_superior`) RLIKE '^0+([.,]0+)?$'),
    'cod_ordem_tem_texto_descritivo', 'string', COUNT_IF(`cod_ordem_tem_texto_descritivo` IS NULL), COUNT_IF(`cod_ordem_tem_texto_descritivo` IS NOT NULL AND lower(trim(`cod_ordem_tem_texto_descritivo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_ordem_tem_texto_descritivo`) RLIKE '^0+([.,]0+)?$'),
    'ind_marcacao_eliminacao', 'string', COUNT_IF(`ind_marcacao_eliminacao` IS NULL), COUNT_IF(`ind_marcacao_eliminacao` IS NOT NULL AND lower(trim(`ind_marcacao_eliminacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_marcacao_eliminacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_rua_endereco', 'string', COUNT_IF(`cod_rua_endereco` IS NULL), COUNT_IF(`cod_rua_endereco` IS NOT NULL AND lower(trim(`cod_rua_endereco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_rua_endereco`) RLIKE '^0+([.,]0+)?$'),
    'cod_regiao_endereco', 'string', COUNT_IF(`cod_regiao_endereco` IS NULL), COUNT_IF(`cod_regiao_endereco` IS NOT NULL AND lower(trim(`cod_regiao_endereco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_regiao_endereco`) RLIKE '^0+([.,]0+)?$'),
    'nm_cidade_endereco', 'string', COUNT_IF(`nm_cidade_endereco` IS NULL), COUNT_IF(`nm_cidade_endereco` IS NOT NULL AND lower(trim(`nm_cidade_endereco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_cidade_endereco`) RLIKE '^0+([.,]0+)?$'),
    'cod_pais_endereco', 'string', COUNT_IF(`cod_pais_endereco` IS NULL), COUNT_IF(`cod_pais_endereco` IS NOT NULL AND lower(trim(`cod_pais_endereco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_pais_endereco`) RLIKE '^0+([.,]0+)?$'),
    'cod_postal_endereco', 'string', COUNT_IF(`cod_postal_endereco` IS NULL), COUNT_IF(`cod_postal_endereco` IS NOT NULL AND lower(trim(`cod_postal_endereco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_postal_endereco`) RLIKE '^0+([.,]0+)?$'),
    'num_telefone_endereco', 'string', COUNT_IF(`num_telefone_endereco` IS NULL), COUNT_IF(`num_telefone_endereco` IS NOT NULL AND lower(trim(`num_telefone_endereco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_telefone_endereco`) RLIKE '^0+([.,]0+)?$')
  ) AS (coluna, tipo, nulos, vazios, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros,
       t.total - p.nulos - p.vazios - p.zeros                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                       THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                              AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade no cenário

In [ ]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'),
card AS (
  SELECT stack(61,
    'cod_ordem', 'string', approx_count_distinct(`cod_ordem`),
    'tp_grupo_planejamento_mnt', 'string', approx_count_distinct(`tp_grupo_planejamento_mnt`),
    'cod_centro_trabalho_principal', 'string', approx_count_distinct(`cod_centro_trabalho_principal`),
    'tp_ordem', 'string', approx_count_distinct(`tp_ordem`),
    'cod_prioridade', 'string', approx_count_distinct(`cod_prioridade`),
    'dt_criacao', 'string', approx_count_distinct(`dt_criacao`),
    'dt_base_inicio', 'string', approx_count_distinct(`dt_base_inicio`),
    'dt_base_fim', 'string', approx_count_distinct(`dt_base_fim`),
    'num_nota', 'string', approx_count_distinct(`num_nota`),
    'desc_breve', 'string', approx_count_distinct(`desc_breve`),
    'cod_revisao', 'string', approx_count_distinct(`cod_revisao`),
    'cod_localizacao', 'string', approx_count_distinct(`cod_localizacao`),
    'tp_grupo_processamento', 'string', approx_count_distinct(`tp_grupo_processamento`),
    'cod_usuario_criacao', 'string', approx_count_distinct(`cod_usuario_criacao`),
    'vl_custo_total_planejado', 'double', approx_count_distinct(`vl_custo_total_planejado`),
    'cod_usuario_modificacao', 'string', approx_count_distinct(`cod_usuario_modificacao`),
    'cod_plano_manutencao', 'string', approx_count_distinct(`cod_plano_manutencao`),
    'cod_equipamento', 'string', approx_count_distinct(`cod_equipamento`),
    'cod_centro_custo_responsavel', 'string', approx_count_distinct(`cod_centro_custo_responsavel`),
    'cod_centro_custo', 'string', approx_count_distinct(`cod_centro_custo`),
    'dt_encerramento_tecnico', 'string', approx_count_distinct(`dt_encerramento_tecnico`),
    'dt_inicio_programado', 'string', approx_count_distinct(`dt_inicio_programado`),
    'dt_fim_programado', 'string', approx_count_distinct(`dt_fim_programado`),
    'dt_referencia', 'string', approx_count_distinct(`dt_referencia`),
    'dh_referencia', 'string', approx_count_distinct(`dh_referencia`),
    'dt_ultima_modificacao', 'string', approx_count_distinct(`dt_ultima_modificacao`),
    'dt_liberacao_real', 'string', approx_count_distinct(`dt_liberacao_real`),
    'dt_fim_confirmado_ordem', 'string', approx_count_distinct(`dt_fim_confirmado_ordem`),
    'dh_base_fim', 'string', approx_count_distinct(`dh_base_fim`),
    'dh_fim_confirmado_ordem', 'string', approx_count_distinct(`dh_fim_confirmado_ordem`),
    'dt_inicio_real', 'string', approx_count_distinct(`dt_inicio_real`),
    'dh_inicio_real', 'string', approx_count_distinct(`dh_inicio_real`),
    'dh_base_inicio', 'string', approx_count_distinct(`dh_base_inicio`),
    'cod_elemento_pep', 'string', approx_count_distinct(`cod_elemento_pep`),
    'cod_pep_ordem', 'string', approx_count_distinct(`cod_pep_ordem`),
    'tp_categoria_ordem', 'string', approx_count_distinct(`tp_categoria_ordem`),
    'cod_centro_planejamento_manutencao', 'string', approx_count_distinct(`cod_centro_planejamento_manutencao`),
    'cod_id_objeto_centro_trabalho', 'string', approx_count_distinct(`cod_id_objeto_centro_trabalho`),
    'tp_grupo_lista_operacoes', 'string', approx_count_distinct(`tp_grupo_lista_operacoes`),
    'cod_variante_lista_operacoes', 'string', approx_count_distinct(`cod_variante_lista_operacoes`),
    'num_serie', 'string', approx_count_distinct(`num_serie`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'desc_produto_material', 'string', approx_count_distinct(`desc_produto_material`),
    'cod_conjunto', 'string', approx_count_distinct(`cod_conjunto`),
    'cod_moeda', 'string', approx_count_distinct(`cod_moeda`),
    'cod_esquema_calculo_custos', 'string', approx_count_distinct(`cod_esquema_calculo_custos`),
    'cod_empresa', 'string', approx_count_distinct(`cod_empresa`),
    'cod_divisao', 'string', approx_count_distinct(`cod_divisao`),
    'cod_centro_lucro', 'string', approx_count_distinct(`cod_centro_lucro`),
    'cod_area_contabilidade_custos', 'string', approx_count_distinct(`cod_area_contabilidade_custos`),
    'cod_ordem_cliente', 'string', approx_count_distinct(`cod_ordem_cliente`),
    'num_item_pedido_venda', 'string', approx_count_distinct(`num_item_pedido_venda`),
    'cod_diagrama_rede_rede_superior', 'string', approx_count_distinct(`cod_diagrama_rede_rede_superior`),
    'cod_ordem_tem_texto_descritivo', 'string', approx_count_distinct(`cod_ordem_tem_texto_descritivo`),
    'ind_marcacao_eliminacao', 'string', approx_count_distinct(`ind_marcacao_eliminacao`),
    'cod_rua_endereco', 'string', approx_count_distinct(`cod_rua_endereco`),
    'cod_regiao_endereco', 'string', approx_count_distinct(`cod_regiao_endereco`),
    'nm_cidade_endereco', 'string', approx_count_distinct(`nm_cidade_endereco`),
    'cod_pais_endereco', 'string', approx_count_distinct(`cod_pais_endereco`),
    'cod_postal_endereco', 'string', approx_count_distinct(`cod_postal_endereco`),
    'num_telefone_endereco', 'string', approx_count_distinct(`num_telefone_endereco`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma, dentro do cenário.

In [ ]:
-- 8. DOMINIO DAS CATEGORICAS
(SELECT 'tp_grupo_planejamento_mnt' AS coluna, CAST(`tp_grupo_planejamento_mnt` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128' GROUP BY `tp_grupo_planejamento_mnt` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_centro_trabalho_principal' AS coluna, CAST(`cod_centro_trabalho_principal` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128' GROUP BY `cod_centro_trabalho_principal` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_ordem' AS coluna, CAST(`tp_ordem` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128' GROUP BY `tp_ordem` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_prioridade' AS coluna, CAST(`cod_prioridade` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128' GROUP BY `cod_prioridade` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_revisao' AS coluna, CAST(`cod_revisao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128' GROUP BY `cod_revisao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_grupo_processamento' AS coluna, CAST(`tp_grupo_processamento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128' GROUP BY `tp_grupo_processamento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_categoria_ordem' AS coluna, CAST(`tp_categoria_ordem` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128' GROUP BY `tp_categoria_ordem` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128' GROUP BY `cod_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_moeda' AS coluna, CAST(`cod_moeda` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128' GROUP BY `cod_moeda` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_marcacao_eliminacao' AS coluna, CAST(`ind_marcacao_eliminacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128' GROUP BY `ind_marcacao_eliminacao` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Perfil dos campos numéricos

Campos `double` exigem tolerância de 0,005 na comparação com o SAP.

In [ ]:
-- 9. PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(1,
    'vl_custo_total_planejado', 'double', COUNT(`vl_custo_total_planejado`), CAST(MIN(`vl_custo_total_planejado`) AS DOUBLE), CAST(MAX(`vl_custo_total_planejado`) AS DOUBLE), CAST(AVG(`vl_custo_total_planejado`) AS DOUBLE), CAST(percentile_approx(`vl_custo_total_planejado`, 0.5) AS DOUBLE), COUNT_IF(`vl_custo_total_planejado` < 0), COUNT_IF(`vl_custo_total_planejado` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
)
ORDER BY coluna;

## 10. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel extraído do SAP e compare linha a linha.
Divergência de total é o teste mais rápido para detectar registro faltando ou duplicado —
e cobre o ponto cego da contagem de linhas, que sozinha não prova nada.

In [ ]:
-- 10. TOTAIS PARA CONCILIACAO
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(1,
    'vl_custo_total_planejado', CAST(SUM(`vl_custo_total_planejado`) AS DOUBLE), CAST(ROUND(SUM(`vl_custo_total_planejado`), 2) AS STRING), COUNT(`vl_custo_total_planejado`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
)
ORDER BY coluna;

## 11. Datas armazenadas como STRING

**Armadilha:** o SAP exporta `2024-02-23 00:00:00` e o Datalake grava `20240223`.
Mesma data, formato diferente — normalizar para `AAAAMMDD` antes de comparar.

In [ ]:
-- 11. DATAS EM STRING
SELECT coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, minimo, maximo,
       CASE WHEN (CASE WHEN fmt_AAAAMMDD > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_ISO       > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_BR        > 0 THEN 1 ELSE 0 END) > 1
            THEN 'ALERTA: mais de um formato' ELSE 'formato unico' END AS veredito
FROM (
  SELECT stack(11,
    'dt_criacao', COUNT_IF(`dt_criacao` IS NULL OR trim(`dt_criacao`) = ''), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_criacao`) NOT IN ('', '00000000') THEN `dt_criacao` END), MAX(CASE WHEN trim(`dt_criacao`) NOT IN ('', '00000000') THEN `dt_criacao` END),
    'dt_base_inicio', COUNT_IF(`dt_base_inicio` IS NULL OR trim(`dt_base_inicio`) = ''), COUNT_IF(trim(`dt_base_inicio`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_base_inicio`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_base_inicio`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_base_inicio`) NOT IN ('', '00000000') THEN `dt_base_inicio` END), MAX(CASE WHEN trim(`dt_base_inicio`) NOT IN ('', '00000000') THEN `dt_base_inicio` END),
    'dt_base_fim', COUNT_IF(`dt_base_fim` IS NULL OR trim(`dt_base_fim`) = ''), COUNT_IF(trim(`dt_base_fim`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_base_fim`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_base_fim`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_base_fim`) NOT IN ('', '00000000') THEN `dt_base_fim` END), MAX(CASE WHEN trim(`dt_base_fim`) NOT IN ('', '00000000') THEN `dt_base_fim` END),
    'dt_encerramento_tecnico', COUNT_IF(`dt_encerramento_tecnico` IS NULL OR trim(`dt_encerramento_tecnico`) = ''), COUNT_IF(trim(`dt_encerramento_tecnico`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_encerramento_tecnico`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_encerramento_tecnico`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_encerramento_tecnico`) NOT IN ('', '00000000') THEN `dt_encerramento_tecnico` END), MAX(CASE WHEN trim(`dt_encerramento_tecnico`) NOT IN ('', '00000000') THEN `dt_encerramento_tecnico` END),
    'dt_inicio_programado', COUNT_IF(`dt_inicio_programado` IS NULL OR trim(`dt_inicio_programado`) = ''), COUNT_IF(trim(`dt_inicio_programado`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_inicio_programado`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_inicio_programado`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_inicio_programado`) NOT IN ('', '00000000') THEN `dt_inicio_programado` END), MAX(CASE WHEN trim(`dt_inicio_programado`) NOT IN ('', '00000000') THEN `dt_inicio_programado` END),
    'dt_fim_programado', COUNT_IF(`dt_fim_programado` IS NULL OR trim(`dt_fim_programado`) = ''), COUNT_IF(trim(`dt_fim_programado`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_fim_programado`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_fim_programado`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_fim_programado`) NOT IN ('', '00000000') THEN `dt_fim_programado` END), MAX(CASE WHEN trim(`dt_fim_programado`) NOT IN ('', '00000000') THEN `dt_fim_programado` END),
    'dt_referencia', COUNT_IF(`dt_referencia` IS NULL OR trim(`dt_referencia`) = ''), COUNT_IF(trim(`dt_referencia`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_referencia`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_referencia`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_referencia`) NOT IN ('', '00000000') THEN `dt_referencia` END), MAX(CASE WHEN trim(`dt_referencia`) NOT IN ('', '00000000') THEN `dt_referencia` END),
    'dt_ultima_modificacao', COUNT_IF(`dt_ultima_modificacao` IS NULL OR trim(`dt_ultima_modificacao`) = ''), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_ultima_modificacao`) NOT IN ('', '00000000') THEN `dt_ultima_modificacao` END), MAX(CASE WHEN trim(`dt_ultima_modificacao`) NOT IN ('', '00000000') THEN `dt_ultima_modificacao` END),
    'dt_liberacao_real', COUNT_IF(`dt_liberacao_real` IS NULL OR trim(`dt_liberacao_real`) = ''), COUNT_IF(trim(`dt_liberacao_real`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_liberacao_real`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_liberacao_real`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_liberacao_real`) NOT IN ('', '00000000') THEN `dt_liberacao_real` END), MAX(CASE WHEN trim(`dt_liberacao_real`) NOT IN ('', '00000000') THEN `dt_liberacao_real` END),
    'dt_fim_confirmado_ordem', COUNT_IF(`dt_fim_confirmado_ordem` IS NULL OR trim(`dt_fim_confirmado_ordem`) = ''), COUNT_IF(trim(`dt_fim_confirmado_ordem`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_fim_confirmado_ordem`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_fim_confirmado_ordem`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_fim_confirmado_ordem`) NOT IN ('', '00000000') THEN `dt_fim_confirmado_ordem` END), MAX(CASE WHEN trim(`dt_fim_confirmado_ordem`) NOT IN ('', '00000000') THEN `dt_fim_confirmado_ordem` END),
    'dt_inicio_real', COUNT_IF(`dt_inicio_real` IS NULL OR trim(`dt_inicio_real`) = ''), COUNT_IF(trim(`dt_inicio_real`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_inicio_real`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_inicio_real`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_inicio_real`) NOT IN ('', '00000000') THEN `dt_inicio_real` END), MAX(CASE WHEN trim(`dt_inicio_real`) NOT IN ('', '00000000') THEN `dt_inicio_real` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, minimo, maximo)
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
)
ORDER BY coluna;

## 12. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

In [ ]:
-- 12. CODIGOS
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(7,
    'cod_ordem', 'string', COUNT_IF(CAST(`cod_ordem` AS STRING) IS NULL OR trim(CAST(`cod_ordem` AS STRING)) = ''), MIN(length(trim(CAST(`cod_ordem` AS STRING)))), MAX(length(trim(CAST(`cod_ordem` AS STRING)))), COUNT_IF(trim(CAST(`cod_ordem` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_ordem` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_ordem` AS STRING)), '^0+', '')),
    'num_nota', 'string', COUNT_IF(CAST(`num_nota` AS STRING) IS NULL OR trim(CAST(`num_nota` AS STRING)) = ''), MIN(length(trim(CAST(`num_nota` AS STRING)))), MAX(length(trim(CAST(`num_nota` AS STRING)))), COUNT_IF(trim(CAST(`num_nota` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_nota` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_nota` AS STRING)), '^0+', '')),
    'cod_equipamento', 'string', COUNT_IF(CAST(`cod_equipamento` AS STRING) IS NULL OR trim(CAST(`cod_equipamento` AS STRING)) = ''), MIN(length(trim(CAST(`cod_equipamento` AS STRING)))), MAX(length(trim(CAST(`cod_equipamento` AS STRING)))), COUNT_IF(trim(CAST(`cod_equipamento` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_equipamento` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_equipamento` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_centro_custo', 'string', COUNT_IF(CAST(`cod_centro_custo` AS STRING) IS NULL OR trim(CAST(`cod_centro_custo` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro_custo` AS STRING)))), MAX(length(trim(CAST(`cod_centro_custo` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro_custo` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro_custo` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro_custo` AS STRING)), '^0+', '')),
    'cod_plano_manutencao', 'string', COUNT_IF(CAST(`cod_plano_manutencao` AS STRING) IS NULL OR trim(CAST(`cod_plano_manutencao` AS STRING)) = ''), MIN(length(trim(CAST(`cod_plano_manutencao` AS STRING)))), MAX(length(trim(CAST(`cod_plano_manutencao` AS STRING)))), COUNT_IF(trim(CAST(`cod_plano_manutencao` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_plano_manutencao` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_plano_manutencao` AS STRING)), '^0+', '')),
    'num_serie', 'string', COUNT_IF(CAST(`num_serie` AS STRING) IS NULL OR trim(CAST(`num_serie` AS STRING)) = ''), MIN(length(trim(CAST(`num_serie` AS STRING)))), MAX(length(trim(CAST(`num_serie` AS STRING)))), COUNT_IF(trim(CAST(`num_serie` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_serie` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_serie` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
)
ORDER BY coluna;

## 13. Chaves normalizadas para join com o SAP

Lista das chaves já **sem zeros à esquerda**, prontas para colar no Excel
e cruzar com o extrato do SAP via PROCV/ÍNDICE.

Baixe como CSV e use para identificar registros presentes de um lado e ausentes do outro.

In [ ]:
-- 13. CHAVES NORMALIZADAS (para cruzar com o SAP)
SELECT DISTINCT
       regexp_replace(trim(CAST(`cod_ordem` AS STRING)), '^0+', '') AS `cod_ordem_norm`
FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
ORDER BY 1;

## 14. Checksum de linha

Gera uma impressão digital de cada linha. Dois usos:

- **Contar linhas realmente distintas** — se `linhas` for maior que `linhas_unicas`,
  existem registros 100% idênticos (duplicata real)
- **Comparação rápida** — aplicando a mesma concatenação no SAP, dá para achar
  divergências sem comparar campo a campo

In [ ]:
-- 14. CHECKSUM DE LINHA
SELECT COUNT(*)                                                 AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_ordem` AS STRING), ''), COALESCE(CAST(`tp_grupo_planejamento_mnt` AS STRING), ''), COALESCE(CAST(`cod_centro_trabalho_principal` AS STRING), ''), COALESCE(CAST(`tp_ordem` AS STRING), ''), COALESCE(CAST(`cod_prioridade` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`dt_base_inicio` AS STRING), ''), COALESCE(CAST(`dt_base_fim` AS STRING), ''), COALESCE(CAST(`num_nota` AS STRING), ''), COALESCE(CAST(`desc_breve` AS STRING), ''), COALESCE(CAST(`cod_revisao` AS STRING), ''), COALESCE(CAST(`cod_localizacao` AS STRING), ''), COALESCE(CAST(`tp_grupo_processamento` AS STRING), ''), COALESCE(CAST(`cod_usuario_criacao` AS STRING), ''), COALESCE(CAST(`vl_custo_total_planejado` AS STRING), ''), COALESCE(CAST(`cod_usuario_modificacao` AS STRING), ''), COALESCE(CAST(`cod_plano_manutencao` AS STRING), ''), COALESCE(CAST(`cod_equipamento` AS STRING), ''), COALESCE(CAST(`cod_centro_custo_responsavel` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`dt_encerramento_tecnico` AS STRING), ''), COALESCE(CAST(`dt_inicio_programado` AS STRING), ''), COALESCE(CAST(`dt_fim_programado` AS STRING), ''), COALESCE(CAST(`dt_referencia` AS STRING), ''), COALESCE(CAST(`dh_referencia` AS STRING), ''), COALESCE(CAST(`dt_ultima_modificacao` AS STRING), ''), COALESCE(CAST(`dt_liberacao_real` AS STRING), ''), COALESCE(CAST(`dt_fim_confirmado_ordem` AS STRING), ''), COALESCE(CAST(`dh_base_fim` AS STRING), ''), COALESCE(CAST(`dh_fim_confirmado_ordem` AS STRING), ''), COALESCE(CAST(`dt_inicio_real` AS STRING), ''), COALESCE(CAST(`dh_inicio_real` AS STRING), ''), COALESCE(CAST(`dh_base_inicio` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`cod_pep_ordem` AS STRING), ''), COALESCE(CAST(`tp_categoria_ordem` AS STRING), ''), COALESCE(CAST(`cod_centro_planejamento_manutencao` AS STRING), ''), COALESCE(CAST(`cod_id_objeto_centro_trabalho` AS STRING), ''), COALESCE(CAST(`tp_grupo_lista_operacoes` AS STRING), ''), COALESCE(CAST(`cod_variante_lista_operacoes` AS STRING), ''), COALESCE(CAST(`num_serie` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_produto_material` AS STRING), ''), COALESCE(CAST(`cod_conjunto` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`cod_esquema_calculo_custos` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_divisao` AS STRING), ''), COALESCE(CAST(`cod_centro_lucro` AS STRING), ''), COALESCE(CAST(`cod_area_contabilidade_custos` AS STRING), ''), COALESCE(CAST(`cod_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_item_pedido_venda` AS STRING), ''), COALESCE(CAST(`cod_diagrama_rede_rede_superior` AS STRING), ''), COALESCE(CAST(`cod_ordem_tem_texto_descritivo` AS STRING), ''), COALESCE(CAST(`ind_marcacao_eliminacao` AS STRING), ''), COALESCE(CAST(`cod_rua_endereco` AS STRING), ''), COALESCE(CAST(`cod_regiao_endereco` AS STRING), ''), COALESCE(CAST(`nm_cidade_endereco` AS STRING), ''), COALESCE(CAST(`cod_pais_endereco` AS STRING), ''), COALESCE(CAST(`cod_postal_endereco` AS STRING), ''), COALESCE(CAST(`num_telefone_endereco` AS STRING), ''))))             AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_ordem` AS STRING), ''), COALESCE(CAST(`tp_grupo_planejamento_mnt` AS STRING), ''), COALESCE(CAST(`cod_centro_trabalho_principal` AS STRING), ''), COALESCE(CAST(`tp_ordem` AS STRING), ''), COALESCE(CAST(`cod_prioridade` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`dt_base_inicio` AS STRING), ''), COALESCE(CAST(`dt_base_fim` AS STRING), ''), COALESCE(CAST(`num_nota` AS STRING), ''), COALESCE(CAST(`desc_breve` AS STRING), ''), COALESCE(CAST(`cod_revisao` AS STRING), ''), COALESCE(CAST(`cod_localizacao` AS STRING), ''), COALESCE(CAST(`tp_grupo_processamento` AS STRING), ''), COALESCE(CAST(`cod_usuario_criacao` AS STRING), ''), COALESCE(CAST(`vl_custo_total_planejado` AS STRING), ''), COALESCE(CAST(`cod_usuario_modificacao` AS STRING), ''), COALESCE(CAST(`cod_plano_manutencao` AS STRING), ''), COALESCE(CAST(`cod_equipamento` AS STRING), ''), COALESCE(CAST(`cod_centro_custo_responsavel` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`dt_encerramento_tecnico` AS STRING), ''), COALESCE(CAST(`dt_inicio_programado` AS STRING), ''), COALESCE(CAST(`dt_fim_programado` AS STRING), ''), COALESCE(CAST(`dt_referencia` AS STRING), ''), COALESCE(CAST(`dh_referencia` AS STRING), ''), COALESCE(CAST(`dt_ultima_modificacao` AS STRING), ''), COALESCE(CAST(`dt_liberacao_real` AS STRING), ''), COALESCE(CAST(`dt_fim_confirmado_ordem` AS STRING), ''), COALESCE(CAST(`dh_base_fim` AS STRING), ''), COALESCE(CAST(`dh_fim_confirmado_ordem` AS STRING), ''), COALESCE(CAST(`dt_inicio_real` AS STRING), ''), COALESCE(CAST(`dh_inicio_real` AS STRING), ''), COALESCE(CAST(`dh_base_inicio` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`cod_pep_ordem` AS STRING), ''), COALESCE(CAST(`tp_categoria_ordem` AS STRING), ''), COALESCE(CAST(`cod_centro_planejamento_manutencao` AS STRING), ''), COALESCE(CAST(`cod_id_objeto_centro_trabalho` AS STRING), ''), COALESCE(CAST(`tp_grupo_lista_operacoes` AS STRING), ''), COALESCE(CAST(`cod_variante_lista_operacoes` AS STRING), ''), COALESCE(CAST(`num_serie` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_produto_material` AS STRING), ''), COALESCE(CAST(`cod_conjunto` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`cod_esquema_calculo_custos` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_divisao` AS STRING), ''), COALESCE(CAST(`cod_centro_lucro` AS STRING), ''), COALESCE(CAST(`cod_area_contabilidade_custos` AS STRING), ''), COALESCE(CAST(`cod_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_item_pedido_venda` AS STRING), ''), COALESCE(CAST(`cod_diagrama_rede_rede_superior` AS STRING), ''), COALESCE(CAST(`cod_ordem_tem_texto_descritivo` AS STRING), ''), COALESCE(CAST(`ind_marcacao_eliminacao` AS STRING), ''), COALESCE(CAST(`cod_rua_endereco` AS STRING), ''), COALESCE(CAST(`cod_regiao_endereco` AS STRING), ''), COALESCE(CAST(`nm_cidade_endereco` AS STRING), ''), COALESCE(CAST(`cod_pais_endereco` AS STRING), ''), COALESCE(CAST(`cod_postal_endereco` AS STRING), ''), COALESCE(CAST(`num_telefone_endereco` AS STRING), ''))))  AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_ordem` AS STRING), ''), COALESCE(CAST(`tp_grupo_planejamento_mnt` AS STRING), ''), COALESCE(CAST(`cod_centro_trabalho_principal` AS STRING), ''), COALESCE(CAST(`tp_ordem` AS STRING), ''), COALESCE(CAST(`cod_prioridade` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`dt_base_inicio` AS STRING), ''), COALESCE(CAST(`dt_base_fim` AS STRING), ''), COALESCE(CAST(`num_nota` AS STRING), ''), COALESCE(CAST(`desc_breve` AS STRING), ''), COALESCE(CAST(`cod_revisao` AS STRING), ''), COALESCE(CAST(`cod_localizacao` AS STRING), ''), COALESCE(CAST(`tp_grupo_processamento` AS STRING), ''), COALESCE(CAST(`cod_usuario_criacao` AS STRING), ''), COALESCE(CAST(`vl_custo_total_planejado` AS STRING), ''), COALESCE(CAST(`cod_usuario_modificacao` AS STRING), ''), COALESCE(CAST(`cod_plano_manutencao` AS STRING), ''), COALESCE(CAST(`cod_equipamento` AS STRING), ''), COALESCE(CAST(`cod_centro_custo_responsavel` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`dt_encerramento_tecnico` AS STRING), ''), COALESCE(CAST(`dt_inicio_programado` AS STRING), ''), COALESCE(CAST(`dt_fim_programado` AS STRING), ''), COALESCE(CAST(`dt_referencia` AS STRING), ''), COALESCE(CAST(`dh_referencia` AS STRING), ''), COALESCE(CAST(`dt_ultima_modificacao` AS STRING), ''), COALESCE(CAST(`dt_liberacao_real` AS STRING), ''), COALESCE(CAST(`dt_fim_confirmado_ordem` AS STRING), ''), COALESCE(CAST(`dh_base_fim` AS STRING), ''), COALESCE(CAST(`dh_fim_confirmado_ordem` AS STRING), ''), COALESCE(CAST(`dt_inicio_real` AS STRING), ''), COALESCE(CAST(`dh_inicio_real` AS STRING), ''), COALESCE(CAST(`dh_base_inicio` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`cod_pep_ordem` AS STRING), ''), COALESCE(CAST(`tp_categoria_ordem` AS STRING), ''), COALESCE(CAST(`cod_centro_planejamento_manutencao` AS STRING), ''), COALESCE(CAST(`cod_id_objeto_centro_trabalho` AS STRING), ''), COALESCE(CAST(`tp_grupo_lista_operacoes` AS STRING), ''), COALESCE(CAST(`cod_variante_lista_operacoes` AS STRING), ''), COALESCE(CAST(`num_serie` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_produto_material` AS STRING), ''), COALESCE(CAST(`cod_conjunto` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`cod_esquema_calculo_custos` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_divisao` AS STRING), ''), COALESCE(CAST(`cod_centro_lucro` AS STRING), ''), COALESCE(CAST(`cod_area_contabilidade_custos` AS STRING), ''), COALESCE(CAST(`cod_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_item_pedido_venda` AS STRING), ''), COALESCE(CAST(`cod_diagrama_rede_rede_superior` AS STRING), ''), COALESCE(CAST(`cod_ordem_tem_texto_descritivo` AS STRING), ''), COALESCE(CAST(`ind_marcacao_eliminacao` AS STRING), ''), COALESCE(CAST(`cod_rua_endereco` AS STRING), ''), COALESCE(CAST(`cod_regiao_endereco` AS STRING), ''), COALESCE(CAST(`nm_cidade_endereco` AS STRING), ''), COALESCE(CAST(`cod_pais_endereco` AS STRING), ''), COALESCE(CAST(`cod_postal_endereco` AS STRING), ''), COALESCE(CAST(`num_telefone_endereco` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128';

## 15. Amostra de linhas completas

In [ ]:
-- 15. AMOSTRA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
ORDER BY `cod_ordem`
LIMIT 20;

## 16. Distribuição interna do centro 4128

Como o volume se reparte dentro do cenário. Útil para conferir se o extrato do SAP
tem a mesma composição.

In [ ]:
-- 16. DISTRIBUICAO POR tp_ordem
SELECT COALESCE(NULLIF(trim(CAST(`tp_ordem` AS STRING)), ''), '(vazio)') AS `tp_ordem`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_ordem` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR cod_empresa
SELECT COALESCE(NULLIF(trim(CAST(`cod_empresa` AS STRING)), ''), '(vazio)') AS `cod_empresa`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_empresa` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR tp_grupo_planejamento_mnt
SELECT COALESCE(NULLIF(trim(CAST(`tp_grupo_planejamento_mnt` AS STRING)), ''), '(vazio)') AS `tp_grupo_planejamento_mnt`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_grupo_planejamento_mnt` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR cod_revisao
SELECT COALESCE(NULLIF(trim(CAST(`cod_revisao` AS STRING)), ''), '(vazio)') AS `cod_revisao`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_revisao` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 17. Freshness

In [ ]:
-- 17. FRESHNESS
-- Tabela sem coluna de data de ingestao. Use o historico de gravacao.
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_ind_iw39 LIMIT 10;

## 18. Análises específicas — IW39

### 18.1 Coerência do fluxo de datas

In [ ]:
-- 18.1 COERENCIA DE DATAS
SELECT 'criacao <= base_inicio' AS regra,
       COUNT_IF(to_date(CASE WHEN `dt_criacao` RLIKE '^[0-9]{8}$' AND `dt_criacao` <> '00000000' THEN `dt_criacao` END, 'yyyyMMdd') IS NOT NULL AND to_date(CASE WHEN `dt_base_inicio` RLIKE '^[0-9]{8}$' AND `dt_base_inicio` <> '00000000' THEN `dt_base_inicio` END, 'yyyyMMdd') IS NOT NULL) AS avaliadas,
       COUNT_IF(to_date(CASE WHEN `dt_criacao` RLIKE '^[0-9]{8}$' AND `dt_criacao` <> '00000000' THEN `dt_criacao` END, 'yyyyMMdd') > to_date(CASE WHEN `dt_base_inicio` RLIKE '^[0-9]{8}$' AND `dt_base_inicio` <> '00000000' THEN `dt_base_inicio` END, 'yyyyMMdd')) AS violacoes
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
UNION ALL
SELECT 'base_inicio <= base_fim' AS regra,
       COUNT_IF(to_date(CASE WHEN `dt_base_inicio` RLIKE '^[0-9]{8}$' AND `dt_base_inicio` <> '00000000' THEN `dt_base_inicio` END, 'yyyyMMdd') IS NOT NULL AND to_date(CASE WHEN `dt_base_fim` RLIKE '^[0-9]{8}$' AND `dt_base_fim` <> '00000000' THEN `dt_base_fim` END, 'yyyyMMdd') IS NOT NULL) AS avaliadas,
       COUNT_IF(to_date(CASE WHEN `dt_base_inicio` RLIKE '^[0-9]{8}$' AND `dt_base_inicio` <> '00000000' THEN `dt_base_inicio` END, 'yyyyMMdd') > to_date(CASE WHEN `dt_base_fim` RLIKE '^[0-9]{8}$' AND `dt_base_fim` <> '00000000' THEN `dt_base_fim` END, 'yyyyMMdd')) AS violacoes
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
UNION ALL
SELECT 'inicio_prog <= fim_prog' AS regra,
       COUNT_IF(to_date(CASE WHEN `dt_inicio_programado` RLIKE '^[0-9]{8}$' AND `dt_inicio_programado` <> '00000000' THEN `dt_inicio_programado` END, 'yyyyMMdd') IS NOT NULL AND to_date(CASE WHEN `dt_fim_programado` RLIKE '^[0-9]{8}$' AND `dt_fim_programado` <> '00000000' THEN `dt_fim_programado` END, 'yyyyMMdd') IS NOT NULL) AS avaliadas,
       COUNT_IF(to_date(CASE WHEN `dt_inicio_programado` RLIKE '^[0-9]{8}$' AND `dt_inicio_programado` <> '00000000' THEN `dt_inicio_programado` END, 'yyyyMMdd') > to_date(CASE WHEN `dt_fim_programado` RLIKE '^[0-9]{8}$' AND `dt_fim_programado` <> '00000000' THEN `dt_fim_programado` END, 'yyyyMMdd')) AS violacoes
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
UNION ALL
SELECT 'inicio_real <= fim_confirmado' AS regra,
       COUNT_IF(to_date(CASE WHEN `dt_inicio_real` RLIKE '^[0-9]{8}$' AND `dt_inicio_real` <> '00000000' THEN `dt_inicio_real` END, 'yyyyMMdd') IS NOT NULL AND to_date(CASE WHEN `dt_fim_confirmado_ordem` RLIKE '^[0-9]{8}$' AND `dt_fim_confirmado_ordem` <> '00000000' THEN `dt_fim_confirmado_ordem` END, 'yyyyMMdd') IS NOT NULL) AS avaliadas,
       COUNT_IF(to_date(CASE WHEN `dt_inicio_real` RLIKE '^[0-9]{8}$' AND `dt_inicio_real` <> '00000000' THEN `dt_inicio_real` END, 'yyyyMMdd') > to_date(CASE WHEN `dt_fim_confirmado_ordem` RLIKE '^[0-9]{8}$' AND `dt_fim_confirmado_ordem` <> '00000000' THEN `dt_fim_confirmado_ordem` END, 'yyyyMMdd')) AS violacoes
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
ORDER BY violacoes DESC;

### 18.2 `cod_revisao` — Grande Parada vs Recorrente

In [ ]:
-- 18.2 COD_REVISAO
SELECT COALESCE(NULLIF(trim(cod_revisao), ''), '(vazio)') AS cod_revisao,
       COUNT(*) AS ordens,
       ROUND(SUM(COALESCE(vl_custo_total_planejado, 0)), 2) AS custo_total
FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
GROUP BY COALESCE(NULLIF(trim(cod_revisao), ''), '(vazio)')
ORDER BY ordens DESC
LIMIT 40;

### 18.3 Completude de vínculos

In [ ]:
-- 18.3 VINCULOS
SELECT 'num_nota' AS vinculo,
       COUNT_IF(`num_nota` IS NULL OR trim(`num_nota`) = '') AS sem_vinculo,
       ROUND(100.0 * COUNT_IF(`num_nota` IS NULL OR trim(`num_nota`) = '') / COUNT(*), 2) AS pct
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
UNION ALL
SELECT 'cod_equipamento' AS vinculo,
       COUNT_IF(`cod_equipamento` IS NULL OR trim(`cod_equipamento`) = '') AS sem_vinculo,
       ROUND(100.0 * COUNT_IF(`cod_equipamento` IS NULL OR trim(`cod_equipamento`) = '') / COUNT(*), 2) AS pct
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
UNION ALL
SELECT 'cod_centro_custo' AS vinculo,
       COUNT_IF(`cod_centro_custo` IS NULL OR trim(`cod_centro_custo`) = '') AS sem_vinculo,
       ROUND(100.0 * COUNT_IF(`cod_centro_custo` IS NULL OR trim(`cod_centro_custo`) = '') / COUNT(*), 2) AS pct
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
UNION ALL
SELECT 'cod_material' AS vinculo,
       COUNT_IF(`cod_material` IS NULL OR trim(`cod_material`) = '') AS sem_vinculo,
       ROUND(100.0 * COUNT_IF(`cod_material` IS NULL OR trim(`cod_material`) = '') / COUNT(*), 2) AS pct
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
UNION ALL
SELECT 'cod_plano_manutencao' AS vinculo,
       COUNT_IF(`cod_plano_manutencao` IS NULL OR trim(`cod_plano_manutencao`) = '') AS sem_vinculo,
       ROUND(100.0 * COUNT_IF(`cod_plano_manutencao` IS NULL OR trim(`cod_plano_manutencao`) = '') / COUNT(*), 2) AS pct
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
UNION ALL
SELECT 'cod_localizacao' AS vinculo,
       COUNT_IF(`cod_localizacao` IS NULL OR trim(`cod_localizacao`) = '') AS sem_vinculo,
       ROUND(100.0 * COUNT_IF(`cod_localizacao` IS NULL OR trim(`cod_localizacao`) = '') / COUNT(*), 2) AS pct
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'
ORDER BY pct DESC;

## 19. EXTRAÇÃO COMPLETA — centro 4128

**Esta é a célula que você baixa para comparar com o SAP.**

Após executar, use **Download → CSV** no resultado.

> **Limites do Databricks:** a tela mostra até 10.000 linhas, mas o download em CSV
> vai além disso. Se o volume for muito grande, use a célula 19.1.

In [ ]:
-- 19. EXTRACAO COMPLETA DO CENARIO
SELECT *
FROM dev_procurement.corp_curated.tbl_ds_ind_iw39
WHERE `cod_centro_planejamento_manutencao` = '4128'
ORDER BY `cod_ordem`;

### 19.1 Alternativa para volume grande _(opcional)_

Descomente para gravar o resultado numa tabela própria e exportar de lá sem limite de tela.

In [ ]:
-- 19.1 GRAVAR EXTRACAO EM TABELA (opcional)
-- CREATE OR REPLACE TABLE dev_procurement.corp_curated.extracao_iw39_4128 AS
-- SELECT * FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128';
--
-- SELECT COUNT(*) FROM dev_procurement.corp_curated.extracao_iw39_4128;
SELECT 'Descomente as linhas acima se precisar gravar a extracao em tabela' AS instrucao;

## 20. Resumo do cenário

Bloco final. **Copie esta saída** e envie ao agente junto com o notebook.

In [ ]:
-- 20. RESUMO DO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128'),
g AS (
  SELECT 'cod_ordem' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_ordem` FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128')
UNION ALL
  SELECT 'cod_ordem + num_nota' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_ordem`, `num_nota` FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128')
UNION ALL
  SELECT 'cod_ordem + cod_equipamento' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_ordem`, `cod_equipamento` FROM dev_procurement.corp_curated.tbl_ds_ind_iw39 WHERE `cod_centro_planejamento_manutencao` = '4128')
)
SELECT 'CENARIO' AS bloco, 'transacao' AS item, 'IW39' AS valor
UNION ALL SELECT 'CENARIO', 'tabela', 'dev_procurement.corp_curated.tbl_ds_ind_iw39'
UNION ALL SELECT 'CENARIO', 'filtro', 'cod_centro_planejamento_manutencao = 4128'
UNION ALL SELECT 'CENARIO', 'linhas no cenario', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'CENARIO', 'colunas', '61'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Baixar a **seção 19** em CSV — é a base do centro 4128 no Datalake.
2. Extrair a mesma transação no SAP com o filtro `centro = 4128`, **todas as abas**.
3. Anotar a data e hora das duas extrações.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Antes de comparar

- [ ] Zeros à esquerda normalizados nos dois lados (seção 12)
- [ ] Formato de data normalizado (seção 11)
- [ ] Totais numéricos conferidos (seção 10)
- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP (seção 6)
